<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](mlcourse.ai) – دورة مفتوحة للتعلم الآلي 
### <center> المؤلف: ألكسندر نيتشيبورينكو، @AlexNich
    
## <center> توقع العملاء الذين سيشترون التأمين على السيارات



### الجزء الأول. شرح الميزات والبيانات



ربما يواجه الكثير منا موقفًا عندما تتصل بك إحدى الشركات لشراء أو شراء شيء ما. أمثلة نموذجية:
* أنت تستخدم بطاقة ائتمان، ويتصل بك البنك ويعرض عليك إصدار قرض؛*
* اشتريت تأميناً على السيارة، وتتصل بك شركة التأمين وتقدم لك أنواعاً أخرى من التأمين؛
* لقد كنت تستخدم الاتصال الخلوي لفترة طويلة، ويتصل بك المشغل الخاص بك مع اقتراح لاستخدام تعريفة جديدة أكثر ربحية (بشكل غريب، وأكثر تكلفة)؛
* اشتريت شيئاً من أحد المتاجر الإلكترونية، وبعد فترة اتصل بك لشراء سلعة أخرى.
* أي حالات تتعلق بالحصول على خدمة جديدة، خدمة إضافية، خدمة أكثر تكلفة.
عادة، في معظم الحالات، لا يوافق العميل على مثل هذه العروض، لأنه ببساطة لا يحتاج إليها. اتضح أن الاتصال بقاعدة العملاء بأكملها طويل وغير فعال، لذلك تحاول الشركات الاتصال فقط بأولئك الذين من المحتمل أن يوافقوا على اقتراحهم. كيف تجد هؤلاء العملاء؟ يمكن القيام بذلك على النحو التالي:
* استدعاء جزء عشوائي معين من العملاء، وتسجيل النتيجة؛
* ابحث في قاعدة العملاء المتبقية عن الأكثر تشابهاً مع أولئك الذين وافقوا على الخدمة المقترحة؛
* الاتصال بهؤلاء العملاء، وبالتالي زيادة فعالية الاتصالات.سوف نقوم بحل مشكلة مماثلة. لدينا مجموعة بيانات من بنك واحد في الولايات المتحدة. إلى جانب الخدمات المعتادة، يوفر هذا البنك أيضًا خدمات التأمين على السيارات. ينظم البنك حملات منتظمة لجذب عملاء جدد. لدى البنك بيانات العملاء المحتملين، ويقوم موظفو البنك بالاتصال بهم للإعلان عن خيارات التأمين المتاحة على السيارات. يتم تزويدنا بمعلومات عامة عن العملاء (العمر، الوظيفة، وما إلى ذلك) بالإضافة إلى معلومات أكثر تحديدًا حول حملة بيع التأمين الحالية (الاتصالات، آخر يوم اتصال) والحملات السابقة (سمات مثل المحاولات السابقة والنتيجة). وتتمثل المهمة في التنبؤ بالعملاء الذين سيشترون التأمين على السيارات أم لا.


In [ ]:
#import libraries

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, GridSearchCV, train_test_split, KFold, learning_curve, validation_curve
from sklearn.metrics import accuracy_score,classification_report,f1_score,roc_auc_score,roc_curve,precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
plt.rcParams['figure.figsize'] = (20,20)
#sns.set(style="darkgrid");
%matplotlib inline
pd.options.display.max_columns=500


دعونا نلقي نظرة على مجموعة البيانات لدينا. يمكنك تحميله من هنا: https://www.kaggle.com/kondla/carinsurance


In [ ]:
data = pd.read_csv('carInsurance_train.csv',index_col='Id')

In [ ]:
data.head()

In [ ]:
data.shape


لدينا 4000 عميل مع 17 ميزة.



المتغير المستهدف لدينا - **'CarInsurance'**، وهو ثنائي (1/0). "1" يعني أن العميل وافق على العرض، و"0" يعني أنه لا.
ثمانية عشر ميزات لمحات عامة:- **المعرف** - رقم المعرف الفريد؛
- **العمر** - عمر العميل؛
- **الوظيفة** - وظيفة العميل.  "المشرف"، "ذوي الياقات الزرقاء"، وما إلى ذلك.
 **الزواجية** - الحالة الاجتماعية للعميل "مطلق"، "متزوج"، "أعزب".
- **التعليم** - المستوى التعليمي للعميل "ابتدائي"، "ثانوي"، إلخ.
- **الافتراضي** - هل يوجد رصيد في حالة التخلف عن السداد؟ "نعم" - 1، "لا" - 0
- **الرصيد** - متوسط الرصيد السنوي بالدولار الأمريكي
- **HHInsurance** - هل الأسرة مؤمنة "نعم" - 1، "لا" - 0
- **قرض السيارة** - هل حصل العميل على قرض سيارة "نعم" - 1، "لا" - 0
- **الاتصالات** - نوع وسيلة الاتصال "خلوي"، "هاتف"، "غير متوفر"
- **LastContactMonth** - شهر آخر جهة اتصال "jan" و"feb" وما إلى ذلك.
- **LastContactDay** - يوم آخر اتصال
- **CallStart** - وقت بدء المكالمة الأخيرة (س س: د د: س س) 12:43:15
- **CallEnd** - وقت انتهاء المكالمة الأخيرة (س س: د د: س س) 12:43:15
- **NoOfContacts** - عدد جهات الاتصال التي تم إجراؤها خلال هذه الحملة لهذا العميل؛ 
- **DaysPassed** - عدد الأيام التي مرت بعد آخر اتصال بالعميل من حملة سابقة (رقم؛ -1 يعني أنه لم يتم الاتصال بالعميل مسبقًا) 
- **المحاولات السابقة** - عدد جهات الاتصال التي تم إجراؤها قبل هذه الحملة ولهذا العميل 
- **النتيجة** - نتيجة الحملة التسويقية السابقة "فشل"، "أخرى"، "نجاح"، "غير موجود".



### الجزء الثاني. تحليل البيانات الأولية



أولاً، افحص بياناتنا حول القيم المفقودة والقيم المتطرفة.


In [ ]:
data.info()

In [ ]:
#devide features in categorical and numerical

data['Default']=data['Default'].astype('object')
data['HHInsurance']=data['HHInsurance'].astype('object')
data['CarLoan']=data['CarLoan'].astype('object')
data['LastContactDay']=data['LastContactDay'].astype('object')

cat = []
num = []
for feature in data.drop(columns=['CarInsurance']).columns:
    if data[feature].dtype == object:
        cat.append(feature)
    else:
        num.append(feature)

In [ ]:
print ('Number of categorical features:',len(cat))
print ('Number of numerical features:',len(num))

In [ ]:
data[data['Job'].isnull()].head()

In [ ]:
data[data['Education'].isnull()].head()

In [ ]:
data[data['Communication'].isnull()].head()

In [ ]:
data[data['Outcome'].isnull()].head()


كما نرى مجموعة البيانات لديها بعض القيم المفقودة: 
* قد يتم تفويت الوظيفة والتعليم لأن العملاء لم يحددوا هذه المعلومات؛
* قد يتم تفويت الاتصال لأن البنك لم يحدد نوع الاتصال
* النتيجة تفتقد إلى القيم لأن بعض العملاء لم يُعرض عليهم أي شيء من قبل، على التوالي، ولا توجد نتيجة؛
سوف نقوم بملء **NaN's** لاحقًا.


In [ ]:
data.describe()

In [ ]:
data.describe(include = ['object'])

تبدو بعض القيم مشبوهة وقد تكون متطرفة:
* **العمر الأقصى = 95 سنة**. الناجي الحقيقي!
* **الرصيد الأقصى = 98,417 دولارًا أمريكيًا**، عندما يكون المتوسط ​​**1532 دولارًا أمريكيًا** ونسبة 75% المئوية تساوي **1619 دولارًا أمريكيًا**. ربما يكون هذا الرجل غنيا جدا؟ إنه أمر نموذجي لتوزيع الدخل.
* **الحد الأدنى للرصيد = - 3058 دولار أمريكي**. ربما أنفق هذا الشخص كل أموال الائتمان ولم يرجع؟
* ** الحد الأقصى لعدد جهات الاتصال = 43 **. هل قدم البنك عدة مرات التأمين داخل هذه الشركة لشخص ما؟ ومن المثير للاهتمام أنه وافق؟
* ** الحد الأقصى للأيام الماضية = 854 **. البنك لا يتصل بشخص ما لأكثر من ثلاث سنوات؟
* **الحد الأقصى لمحاولات PrevAttempts = 58** عندما يكون المتوسط ​​0.72. 
دعونا نلقي نظرة على المعرف بهذه القيم الغريبة.


In [ ]:
data[data['Age']==95].head()

In [ ]:
data[data['Balance']==98417].head()

In [ ]:
data[data['Balance']==-3058].head()

In [ ]:
data[data['DaysPassed']==854].head()

In [ ]:
data[data['NoOfContacts']==43].head()

In [ ]:
data[data['PrevAttempts']==58].head()


بالنظر إلى هذه البيانات، من المستحيل القول أن هناك بالتأكيد بعض الأخطاء في البيانات. ربما كل شيء صحيح. سنقوم لاحقًا بتصور البيانات ونقرر ما يجب فعله بالقيم المشبوهة.



دعونا نرى جزء من العملاء الذين اشتروا التأمين على السيارات.


In [ ]:
data['CarInsurance'].mean()


**40%** ليس سيئا! لكنني أعتقد أن البنك يريد **100%**، لذا فهو يتصل بالعملاء عدة مرات. في مصطلحات ML يمكننا القول أن فئتينا متوازنتان.



الآن قم بفحص تأثير ميزاتنا على المتغير المستهدف. أولاً: الخصائص العددية.


In [ ]:
data.columns

In [ ]:
data.groupby(by=['CarInsurance'])[['Age']].agg([np.mean,np.std,np.min,np.max])

In [ ]:
data.groupby(by=['CarInsurance'])[['NoOfContacts']].agg([np.mean,np.std,np.min,np.max])

In [ ]:
data.groupby(by=['CarInsurance'])[['DaysPassed']].agg([np.mean,np.std,np.min,np.max])

In [ ]:
data.groupby(by=['CarInsurance'])[['PrevAttempts']].agg([np.mean,np.std,np.min,np.max])

In [ ]:
data.groupby(by=['CarInsurance'])[['Balance']].agg([np.mean,np.std,np.min,np.max])


في الجداول التي تم وضعها، يمكننا أن نرى أن هؤلاء العملاء الذين يوافقون على التأمين في المتوسط:
* يقدم البنك المزيد من العروض مع هذا التأمين
* تم عرض عرض على هؤلاء العملاء من قبل شركة بنكية أخرى في المتوسط منذ أكثر من شهرين، بالنسبة لأولئك الذين لم يوافقوا - ما يزيد قليلاً عن شهر
* عرضت عليهم في كثير من الأحيان عروض بنكية أخرى
* احصل على المزيد من التوازن
* لديك اتصالات أقل من البنك للحملات الأخرىلتأكيد هذه الملاحظات، قمنا ببناء رسوم بيانية ومخططات مربعة للميزات بشكل أكبر.
الآن نلقي نظرة على الميزات الفئوية والثنائية.


In [ ]:
pd.crosstab(data['Education'],data['Job'],values=data['CarInsurance'],aggfunc='mean',margins=True)

In [ ]:
pd.crosstab(data['Marital'],data['Education'],values=data['CarInsurance'],aggfunc='mean',margins=True)

In [ ]:
pd.crosstab(data['Default'],data['Job'],values=data['CarInsurance'],aggfunc='mean',margins=True)

In [ ]:
pd.crosstab(data['CarLoan'],data['Job'],values=data['CarInsurance'],aggfunc='mean',margins=True)

In [ ]:
pd.crosstab(data['CarLoan'],data['HHInsurance'],values=data['CarInsurance'],aggfunc='mean',margins=True)

In [ ]:
pd.crosstab(data['Communication'],data['Outcome'],values=data['CarInsurance'],aggfunc='mean',margins=True)

In [ ]:
pd.crosstab(data['Communication'],data['LastContactMonth'],values=data['CarInsurance'],aggfunc='mean',margins=True)

In [ ]:
pd.crosstab(data['LastContactDay'],data['LastContactMonth'],values=data['CarInsurance'],aggfunc='mean',margins=True)


وبالنظر إلى هذه الجداول الترافقية يمكننا أن نرى:
* طريقة الاتصال لا تؤثر على متغير الهدف
* الاعتماد الشهري واليومي للحملة
* الأشخاص الذين لديهم CarLoan نادرون يوافقون على العرض
* الأشخاص الذين لديهم شركة HHInsurance نادرًا ما يوافقون على العرض
* الأشخاص الذين وافقوا على العروض الأخرى للبنك في كثير من الأحيان يوافقون على التأمين
* الأشخاص ذوي التقصير النادر يوافقون على العرض
* الأشخاص العازبون والأشخاص الحاصلون على التعليم العالي يوافقون على التأمين 



### الجزء 3. تحليل البيانات المرئية الأولية



لنقم بعمل تصورات لميزاتنا وتأثيرها على المتغير المستهدف.


In [ ]:
#target distribution
sns.countplot(data['CarInsurance'],palette="Accent");
plt.title('Target distribution');

In [ ]:
#distribution of categorical features

plt.figure(figsize=(20,20))
for i in range(1,len(cat[:11])):
    plt.subplot(4,3,i)
    sns.countplot(data[cat[i-1]],palette='Accent')
    plt.xticks(rotation=90)


يمكن ملاحظة أن بعض قيم الميزات الفئوية (**"الافتراضي=1"** أو الأشهر) تحتوي على عدد صغير من الأمثلة. بشكل عام، عادةً ما يتم دمج هذه القيم في مجموعة واحدة لمنع التجهيز الزائد، وفي الحالة الثنائية، يمكن حذف هذا العمود.


In [ ]:
#target variable versus categorical

plt.figure(figsize=(20,20))
for i in range(1,len(cat[:11])):
    plt.subplot(4,3,i)
    sns.barplot(data[cat[i-1]],data['CarInsurance'],palette='Accent')
    plt.xticks(rotation=90)


تم تأكيد الاستنتاجات المتعلقة باعتماد المتغير المستهدف على الميزات الفئوية التي تم الحصول عليها باستخدام تحليل البيانات الأولية من خلال هذه التصورات (انظر **الجزء 2**).


In [ ]:
#histograms of numerical features and their scatterplots

sns.pairplot(data[num], palette="Accent");

In [ ]:
corr_matrix = data[num].corr()

In [ ]:
sns.heatmap(corr_matrix,cmap="Accent");


من مخططات التشتت والخريطة الحرارية، من الواضح أن علاقاتنا الرقمية ليس لها ارتباطات مرئية، والتوزيعات منحرفة بشدة نحو اليسار باستثناء العمر.


In [ ]:
#histograms of numerical features and their scatterplot

sns.pairplot(data[num + ['CarInsurance']],hue='CarInsurance',palette="Accent",diag_kind='kde');

In [ ]:
#boxplots depending on the target variable

plt.figure(figsize=(20,10))
for i in range(1,len(num)+1):
    plt.subplot(2,3,i)
    sns.boxplot(data=data, x=data['CarInsurance'],y=data[num[i-1]],palette="Accent")

In [ ]:
#graphs depending on the target variable with a limit of 0.975 quantile for better visibility

plt.figure(figsize=(20,10))
for i in range(1,len(num)+1):
    plt.subplot(2,3,i)
    sns.boxplot(data=data, x=data['CarInsurance'],y=data[data[num[i-1]]<data[num[i-1]].quantile(0.975)][num[i-1]],palette="Accent")


وبشكل عام، فإن جميع الاستنتاجات والمؤثرات تتفق أيضًا مع ما تم الحصول عليه نتيجة التحليل في **الجزء الثاني**.



### الجزء 4. الرؤى والتبعيات التي تم العثور عليها 



دعونا نلخص ما هي الأنماط التي تم اكتشافها:* التعليم الثالثي يزيد من فرص قبول عرض التأمين، وقد يكون هؤلاء الأشخاص أكثر مسؤولية وحكمة؛
* الأشخاص الذين ليس لديهم قرض السيارة والتأمين على المنزل أكثر ولاءً لعرض التأمين على السيارات، لكن الأمر يبدو غريباً بعض الشيء
* الأشخاص الذين تقدموا بعروض البنوك الأخرى هم الأكثر ولاءً لعرض التأمين على السيارات؛
* إذا قدم البنك التأمين عدة مرات، فمن المرجح أن يوافق العميل عليه؛
* من المحتمل جدًا أن يكون الأشخاص الذين تم الاتصال بهم آخر مرة في مارس وسبتمبر وأكتوبر وديسمبر قد وافقوا على العرض. قد يكون هذا بسبب موسمية مبيعات السيارات. عادة ما يقوم بتأمين السيارات الجديدة، وفي هذه الأشهر يقوم التجار بعمل خصومات جيدة على السيارات؛
* غالبًا ما يشتري الأشخاص غير المتزوجين تأمينًا على السيارات، ربما يكون لديهم أموال إضافية لهذه الخدمة. المتزوجون ينفقون أموالهم في أشياء أخرى؛
* الأشخاص الذين يشترون التأمين على السيارات لديهم رصيد قليل
* الأشخاص الذين لم يقدم لهم البنك أبدًا خدماته الأخرى هم أقل عرضة للموافقة على التأمين على السيارات. هؤلاء هم العملاء الجدد الذين لم يقم البنك ببناء علاقة معهم بعد.
* الطلاب في كثير من الأحيان شراء التأمين على السيارات. أعتقد أنهم مبتدئون في القيادة لذا فهم بحاجة إلى التأمين.



### الجزء الخامس. اختيار المقاييس


لنفترض أن لدينا بيانات عن **4000** عميل، وهذا يمثل ** 10% ** من قاعدة البيانات بأكملها. وإذا افترضنا أن فعالية المكالمات لبقية العملاء ستكون هي نفسها تقريباً، فإن البنك مهتم بالاتصال بجميع العملاء الذين يوافقون على التأمين لعدد أقل من المكالمات أو عدد المكالمات التي يمكن للبنك إجراؤها. وبالتالي، سيكون من الممكن اختيار المقياس **recall@topK%**. كمقياس، **K** في هذه الحالة سيكون مساويًا لحوالي **50%**. بشكل عام، قد تختلف استراتيجية وقدرات البنك، لذلك يجب أن يكون لديك مصنف عالمي. بشكل عام، قد تختلف استراتيجية وقدرات البنك، لذلك يجب أن يكون لديك مصنف عالمي. في هذه الحالة، المقياس العالمي لمصنفات جودة العمل هو **ROC-AUC**. سوف نستخدمها. في هذه الحالة، يمكننا اختيار العتبة وحساب **K** (أي جزء من العملاء لديه احتمالية أعلى) و**recall**@**topK%**.



### الجزء السادس. اختيار النموذج



توجد في مجموعة البيانات الخاصة بنا ميزات رقمية ذات قيم كبيرة جدًا، ولكنها لا تتعارض مع أي شيء، لذلك سنتركها دون تغيير، ونستخدم **XGBoost** كنموذج للتنبؤ، وهو لا يخشى مثل هذه القيم المتطرفة. كما تتمتع هذه الخوارزمية بأفضل أداء في معظم المهام. كما أن هذه المهمة لا ترتبط بالمخاطر المالية، لذا يمكننا إنشاء "الصندوق الأسود".



### الأجزاء 7-9.المعالجة المسبقة للبيانات. التحقق من الصحة وتعديل المعلمات الفائقة للنموذج. إنشاء ميزات جديدة ووصف هذه العملية.


أولاً، دعونا نملأ **NaN**. لقد افترضنا أن بعض الأشخاص لم يملأوا حقول **التعليم** و**الوظيفة** لأي سبب من الأسباب، لذلك بدلاً من التمريرات، نضع "غير معروف"، وسنفعل الشيء نفسه مع نوع الاتصال. القيم المفقودة في الميزة **النتيجة** سوف نقوم بملئها بـ "no_outcome". بشكل عام، نشير ببساطة إلى القيم المفقودة كفئة أخرى.


In [ ]:
data['Education'].fillna('unknown',inplace=True)
data['Job'].fillna('unknown',inplace=True)
data['Communication'].fillna('unknown',inplace=True)
data['Outcome'].fillna('no_outcome',inplace=True)

In [ ]:
data.head()


لتشفير ميزاتنا الفئوية، سنستخدم الطريقة الشائعة **OHE** باستخدام **pd.get_dummies**.


In [ ]:
data=pd.concat([data.drop(columns=['Job','Marital','Education','Communication','LastContactMonth','Outcome']),pd.get_dummies(data[['Job','Marital','Education','Communication','LastContactMonth','Outcome']])],axis=1)

In [ ]:
data.head()

In [ ]:
data.shape


في البداية، لن نستخدم خاصيتي "CallStart" و"CallEnd"، لأننا نحتاج إلى العمل عليهما وصنع ميزات جديدة منهما.


In [ ]:
# get X and y

X = data.drop(columns=['CallStart','CallEnd','CarInsurance'])
X=X.astype('float')
y = data['CarInsurance']


قم بتقسيم مجموعة البيانات الخاصة بنا بالقطار والأجزاء الصالحة. سوف نستخدم **25%** للتحقق من الصحة. نظرًا لأن لدينا مهمة تصنيف متوازنة، فلن نستخدم التقسيم الطبقي.


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=33)

In [ ]:
#part of class "1" 
y_train.mean(), y_valid.mean()


دعونا نتحقق من جودة XGBoost عبر السيرة الذاتية مع 5 طيات خلط.


In [ ]:
xgb = XGBClassifier(random_state=33, n_jobs=4)
kf = KFold(random_state=33,n_splits=5,shuffle=True)
print ('Mean ROC-AUC CV score:', np.mean(cross_val_score(xgb, X_train, y_train, scoring='roc_auc',cv=kf)))


حسنًا، الآن نحاول إضافة بعض الميزات الإضافية. يمكننا إضافة يوم من أيام الأسبوع، لكن للأسف لا نعرف العام الذي تم فيه إجراء المكالمات. ولذلك سنعمل مع العلامات المرتبطة بوقت المكالمة:
* ساعة بداية المكالمة
* بداية دقيقة المكالمة
* مدة المكالمة بالثواني


In [ ]:
data['CallDuration']=pd.to_datetime(data['CallEnd'])-pd.to_datetime(data['CallStart'])
data['CallDuration']=data['CallDuration'].dt.total_seconds()
data['CallHourStart']=pd.to_datetime(data['CallStart']).apply(lambda t: t.hour)
data['CallMinStart']=pd.to_datetime(data['CallStart']).apply(lambda t: t.minute)

In [ ]:
data.head()


دعونا نلقي نظرة على الميزات الجديدة لدينا.


In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=data, x=data['CarInsurance'],y=data['CallDuration'],palette="Accent");

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data['CallHourStart'],data['CarInsurance'],palette='Accent');

In [ ]:
plt.figure(figsize=(15,6))
sns.barplot(data['CallMinStart'],data['CarInsurance'],palette='Accent');
plt.xticks(rotation=90);


**"CallDurations"** هي ميزة مفيدة للغاية، فالمكالمات الطويلة تؤدي إلى شراء التأمين. لا تبدو الميزات الأخرى مفيدة كثيرًا، ولكننا سنحاول تجربتها جميعًا معًا.


In [ ]:
# get X and y

X = data.drop(columns=['CallStart','CallEnd','CarInsurance'])
X=X.astype('float')
y = data['CarInsurance']

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=33)

In [ ]:
#part of class "1" 
y_train.mean(), y_valid.mean()


دعونا نتحقق من الجودة مرة أخرى.


In [ ]:
xgb = XGBClassifier(random_state=33, n_jobs=4)
kf = KFold(random_state=33,n_splits=5,shuffle=True)
print ('Mean ROC-AUC CV score:', np.mean(cross_val_score(xgb, X_train, y_train, scoring='roc_auc',cv=kf)))


قف! الميزات الجديدة أعطت زيادة ملحوظة في الجودة! دعونا نضبط المعلمات الفائقة عبر GridSearchCV.


In [ ]:
%%time
parameters = {'n_estimators':[40, 50, 60, 80, 100, 150, 200, 300], 'max_depth':[3, 4, 5, 6, 7, 8], 'min_child_weight': [1,3,5,7,9]}
xgb = XGBClassifier(random_state=33, n_jobs=4)
clf = GridSearchCV(xgb, parameters, scoring='roc_auc', cv=kf)
clf.fit(X_train, y_train)
print('Best parameters: ', clf.best_params_)


تحقق الآن من المعلمات الفائقة الجديدة عبر سيرتنا الذاتية.


In [ ]:
xgb = XGBClassifier(random_state=33, n_jobs=4,max_depth=4, min_child_weight=1, n_estimators=200)
kf = KFold(random_state=33,n_splits=5,shuffle=True)
print ('Mean ROC-AUC CV score:', np.mean(cross_val_score(xgb, X_train, y_train, scoring='roc_auc',cv=kf)))


والنتيجة الآن أعلى.



### الجزء العاشر. رسم منحنيات التدريب والتحقق من الصحة


In [ ]:
def plot_with_std(x, data, **kwargs):
        mu, std = data.mean(1), data.std(1)
        lines = plt.plot(x, mu, '-', **kwargs)
        plt.fill_between(x, mu - std, mu + std, edgecolor='none',
                         facecolor=lines[0].get_color(), alpha=0.2)
        
def plot_learning_curve(clf, X, y, scoring, cv=5):
 
    train_sizes = np.linspace(0.05, 1, 20)
   
    n_train, val_train, val_test = learning_curve(clf, X=X, y=y, train_sizes=train_sizes, cv=cv,scoring=scoring)
    plot_with_std(n_train, val_train, label='training scores', c='green')
    plot_with_std(n_train, val_test, label='validation scores', c='red')
    plt.xlabel('Training Set Size'); plt.ylabel(scoring)
    plt.legend()

def plot_validation_curve(clf, X, y, cv_param_name, 
                          cv_param_values, scoring):

    val_train, val_test = validation_curve(clf, X, y, cv_param_name, cv_param_values, cv=5, scoring=scoring)
    plot_with_std(cv_param_values, val_train, 
                  label='training scores', c='green')
    plot_with_std(cv_param_values, val_test, 
                  label='validation scores', c='red')
    plt.xlabel(cv_param_name); plt.ylabel(scoring)
    plt.legend()

In [ ]:
# learning curve
plt.figure(figsize=(12,6))
plot_learning_curve(xgb,X_train, y_train, scoring='roc_auc', cv=10)

بالنظر إلى منحنى التعلم، يمكننا القول أن إضافة البيانات يمكن أن يحسن جودة النماذج، لأنه مع إضافة بيانات جديدة، تتزايد جودة التحقق من الصحة.


In [ ]:
# validation curve

plt.figure(figsize=(12,6))
max_depth = [3, 4, 5, 6, 7, 8]
plot_validation_curve(XGBClassifier(random_state=33, n_jobs=4, min_child_weight=1, n_estimators=200), X_train, y_train, 
                    cv_param_name='max_depth', 
                    cv_param_values=max_depth,
                    scoring='roc_auc')

In [ ]:
# validation curve

plt.figure(figsize=(12,6))
n_estimators = [40, 50, 60, 80, 100, 150, 200, 300]
plot_validation_curve(XGBClassifier(random_state=33, n_jobs=4, min_child_weight=1, n_estimators=200), X_train, y_train, 
                    cv_param_name='n_estimators', 
                    cv_param_values=n_estimators,
                    scoring='roc_auc')

In [ ]:
# validation curve

plt.figure(figsize=(12,6))
min_child_weight = [1,3,5,7,9]
plot_validation_curve(XGBClassifier(random_state=33, n_jobs=4, min_child_weight=1, n_estimators=200), X_train, y_train, 
                    cv_param_name='min_child_weight', 
                    cv_param_values=min_child_weight,
                    scoring='roc_auc')


تظهر منحنيات التحقق من الصحة أن النتيجة في السيرة الذاتية أقل بكثير منها في القطار. يشير هذا إلى تجاوز النموذج. بالنسبة لمجموعة البيانات الصغيرة هذه، يعد التعزيز أمرًا شائعًا. لتقليل درجة التجهيز الزائد، يمكنك محاولة تقليل تعقيد النموذج وزيادة المعلمات المسؤولة عن التنظيم.



### الجزء 11. التنبؤ بالعينات الاختبارية أو المحتجزة



الآن نستخدم XGBoost الخاص بنا للتنبؤ باحتمالات X_valid الخاصة بنا.


In [ ]:
xgb.fit(X_train,y_train)
y_pred_valid=xgb.predict_proba(X_valid)[:,1]

print ('ROC-AUC score of X_valid:', roc_auc_score(y_valid, y_pred_valid))


لقد حصلنا على درجة أعلى قليلاً من تلك التي حصلنا عليها في السيرة الذاتية، وهذا يعني أن سيرتنا الذاتية صحيحة.


In [ ]:
import xgboost

In [ ]:
#look at most important features
xgboost.plot_importance(xgb,max_num_features=15,importance_type='gain');


### الجزء 12. الاستنتاجات



في هذا المشروع قمنا بتصميم نموذج بجودة جيدة **~0.92 ROC-AUC**، بحيث يمكن للبنك استخدامه للعثور على العملاء الذين من المرجح أن يشتروا التأمين على السيارات، اعتمادًا على قدرات البنك وسياساته.